# 04 Train WLASL1000 BiGRU + Temporal Attention Model

## Purpose
This notebook trains the selected **BiGRU + Temporal Attention V1** architecture on WLASL1000.

## Why V1?
WLASL300 V2 was worse than V1, so WLASL1000 uses the stable model:

```text
keypoints + velocity
input shape = (60, 516)
BiGRU + Temporal Attention
```

In [3]:
import sys
print(sys.executable)

e:\Be_My_Ear\.venv\Scripts\python.exe


In [4]:
from pathlib import Path
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")
DATASET_NAME = "WLASL1000"
PREFIX = "wlasl1000"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / f"bigru_attention_{PREFIX}.pt"
HISTORY_PATH = MODEL_DIR / f"bigru_attention_{PREFIX}_history.csv"
NORM_STATS_PATH = MODEL_DIR / f"{PREFIX}_train_norm_stats.npz"

BATCH_SIZE = 32
USE_VELOCITY = True
INPUT_SIZE = 516
SEQUENCE_LENGTH = 60
EPOCHS = 70
EARLY_STOPPING_PATIENCE = 15

df = pd.read_csv(CLEAN_INDEX_FILE)

print("Dataset:", DATASET_NAME)
print("Clean samples:", len(df))
print("Classes:", df["label_id"].nunique())
print("Model will save to:", MODEL_PATH)

e:\Be_My_Ear\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: WLASL1000
Clean samples: 7232
Classes: 1000
Model will save to: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_wlasl1000.pt


## 1. Create train / validation / test split

In [5]:
train_records = []
val_records = []
test_records = []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)

    n = len(group)
    n_test = max(1, int(round(n * 0.15)))
    n_val = max(1, int(round(n * 0.15)))

    test_records.append(group.iloc[:n_test])
    val_records.append(group.iloc[n_test:n_test + n_val])
    train_records.append(group.iloc[n_test + n_val:])

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Train classes:", train_df["label_id"].nunique())
print("Validation classes:", val_df["label_id"].nunique())
print("Test classes:", test_df["label_id"].nunique())

Train samples: 5024
Validation samples: 1104
Test samples: 1104
Train classes: 999
Validation classes: 1000
Test classes: 1000


## 2. Compute train-set normalisation

In [6]:
def compute_train_normalisation_stats(train_dataframe):
    total_sum = None
    total_sq_sum = None
    total_count = 0

    for path in tqdm(train_dataframe["keypoint_path"], desc="Computing train mean/std"):
        arr = np.load(path).astype(np.float32)

        if total_sum is None:
            total_sum = arr.sum(axis=0)
            total_sq_sum = (arr ** 2).sum(axis=0)
        else:
            total_sum += arr.sum(axis=0)
            total_sq_sum += (arr ** 2).sum(axis=0)

        total_count += arr.shape[0]

    mean = total_sum / total_count
    variance = (total_sq_sum / total_count) - (mean ** 2)
    variance = np.maximum(variance, 1e-6)
    std = np.sqrt(variance)

    return mean.astype(np.float32), std.astype(np.float32)

train_mean, train_std = compute_train_normalisation_stats(train_df)
np.savez(NORM_STATS_PATH, mean=train_mean, std=train_std)

print("Saved normalisation stats:", NORM_STATS_PATH)

Computing train mean/std: 100%|██████████| 5024/5024 [00:07<00:00, 664.77it/s]

Saved normalisation stats: E:\Be_My_Ear\models\ASL\WLASL1000\wlasl1000_train_norm_stats.npz


## 3. Create dataset, loaders, and model

In [7]:
class SignKeypointDataset(Dataset):
    def __init__(self, dataframe, mean, std, use_velocity=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)
        self.use_velocity = use_velocity

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        keypoints = np.load(row["keypoint_path"]).astype(np.float32)
        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        if self.use_velocity:
            velocity = np.zeros_like(keypoints, dtype=np.float32)
            velocity[1:] = keypoints[1:] - keypoints[:-1]
            features = np.concatenate([keypoints, velocity], axis=1)
        else:
            features = keypoints

        return torch.tensor(features, dtype=torch.float32), torch.tensor(int(row["label_id"]), dtype=torch.long)


train_dataset = SignKeypointDataset(train_df, train_mean, train_std, USE_VELOCITY)
val_dataset = SignKeypointDataset(val_df, train_mean, train_std, USE_VELOCITY)
test_dataset = SignKeypointDataset(test_df, train_mean, train_std, USE_VELOCITY)

class_counts = train_df["label_id"].value_counts().to_dict()
sample_weights = train_df["label_id"].map(lambda label: 1.0 / class_counts[label]).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))
print("Input batch shape:", x_batch.shape)


class BiGRUAttentionModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.4):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)
        scores = self.attention(gru_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = torch.sum(gru_out * weights, dim=1)
        return self.classifier(context)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = df["label_id"].nunique()

model = BiGRUAttentionModel(INPUT_SIZE, 256, NUM_CLASSES, num_layers=2, dropout=0.4).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

print("Using device:", device)
print("Number of classes:", NUM_CLASSES)

Input batch shape: torch.Size([32, 60, 516])
Using device: cuda
Number of classes: 1000


## 4. Training helpers

In [8]:
def top_k_accuracy(outputs, labels, k=5):
    _, top_k_preds = outputs.topk(k, dim=1)
    correct = top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds))
    return correct.any(dim=1).float().mean().item()


def run_epoch(model, loader, optimizer=None, phase="Train", epoch=1, total_epochs=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = total_top1 = total_top3 = total_top5 = 0
    all_preds, all_labels = [], []

    progress_bar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [{phase}]", leave=False)

    with torch.set_grad_enabled(is_train):
        for step, (x, y) in enumerate(progress_bar, start=1):
            x, y = x.to(device), y.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(x)
            loss = criterion(outputs, y)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item()
            total_top1 += (preds == y).float().mean().item()
            total_top3 += top_k_accuracy(outputs, y, k=3)
            total_top5 += top_k_accuracy(outputs, y, k=5)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(y.detach().cpu().numpy())

            progress_bar.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{loss.item():.4f}",
                "top1": f"{(preds == y).float().mean().item():.4f}",
                "top5": f"{top_k_accuracy(outputs, y, k=5):.4f}"
            })

    return (
        total_loss / len(loader),
        total_top1 / len(loader),
        total_top3 / len(loader),
        total_top5 / len(loader),
        f1_score(all_labels, all_preds, average="macro", zero_division=0)
    )

## 5. Train WLASL1000 model

In [9]:
history = {k: [] for k in [
    "train_loss", "train_top1", "train_top3", "train_top5", "train_f1",
    "val_loss", "val_top1", "val_top3", "val_top5", "val_f1", "lr"
]}

best_val_f1 = 0.0
best_val_top5 = 0.0
epochs_without_improvement = 0

print("=" * 80)
print("Be My Ear - WLASL1000 BiGRU + Temporal Attention Training")
print("=" * 80)
print(f"Device: {device}")
print(f"Input shape: (60, {INPUT_SIZE})")
print(f"Classes: {NUM_CLASSES}")
print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Model save path: {MODEL_PATH}")
print("=" * 80)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"\\nEpoch {epoch}/{EPOCHS}")
    print("-" * 80)

    train_loss, train_top1, train_top3, train_top5, train_f1 = run_epoch(
        model, train_loader, optimizer=optimizer, phase="Training", epoch=epoch, total_epochs=EPOCHS
    )

    val_loss, val_top1, val_top3, val_top5, val_f1 = run_epoch(
        model, val_loader, optimizer=None, phase="Validation", epoch=epoch, total_epochs=EPOCHS
    )

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_top1"].append(train_top1)
    history["train_top3"].append(train_top3)
    history["train_top5"].append(train_top5)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top3"].append(val_top3)
    history["val_top5"].append(val_top5)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_top5 = val_top5
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1": best_val_f1,
            "best_val_top5": best_val_top5,
            "num_classes": NUM_CLASSES,
            "input_size": INPUT_SIZE,
            "sequence_length": SEQUENCE_LENGTH,
            "use_velocity": USE_VELOCITY,
            "architecture": "BiGRUAttentionModel"
        }, MODEL_PATH)

        status = "Saved new best model"
    else:
        epochs_without_improvement += 1
        status = "No improvement"

    print(f"Train | Loss: {train_loss:.4f} | Top-1: {train_top1:.4f} | Top-3: {train_top3:.4f} | Top-5: {train_top5:.4f} | F1: {train_f1:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Top-1: {val_top1:.4f} | Top-3: {val_top3:.4f} | Top-5: {val_top5:.4f} | F1: {val_f1:.4f}")
    print(f"Learning rate: {current_lr:.8f}")
    print(f"Status: {status}")
    print(f"Best Val F1 so far: {best_val_f1:.4f}")
    print(f"Best Val Top-5 so far: {best_val_top5:.4f}")
    print(f"Epochs without improvement: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\\nEarly stopping triggered.")
        break

print("\\nTraining time minutes:", round((time.time() - start_time) / 60, 2))
print("Best model saved to:", MODEL_PATH)

Be My Ear - WLASL1000 BiGRU + Temporal Attention Training
Device: cuda
Input shape: (60, 516)
Classes: 1000
Train samples: 5024
Validation samples: 1104
Test samples: 1104
Epochs: 70
Batch size: 32
Model save path: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_wlasl1000.pt
\nEpoch 1/70
--------------------------------------------------------------------------------


Train | Loss: 6.8997 | Top-1: 0.0032 | Top-3: 0.0084 | Top-5: 0.0119 | F1: 0.0006
Val   | Loss: 6.7256 | Top-1: 0.0027 | Top-3: 0.0098 | Top-5: 0.0116 | F1: 0.0001
Learning rate: 0.00030000
Status: Saved new best model
Best Val F1 so far: 0.0001
Best Val Top-5 so far: 0.0116
Epochs without improvement: 0/15
\nEpoch 2/70
--------------------------------------------------------------------------------


Train | Loss: 6.5277 | Top-1: 0.0115 | Top-3: 0.0261 | Top-5: 0.0364 | F1: 0.0053
Val   | Loss: 6.4558 | Top-1: 0.0116 | Top-3: 0.0223 | Top-5: 0.0348 | F1: 0.0022
Learning rate: 0.00030000
Status: Saved new best model
Best Val F1 so far: 0.0022
Best Val Top-5 so far: 0.0348
Epochs without improvement: 0/15
\nEpoch 3/70
--------------------------------------------------------------------------------


Train | Loss: 6.1350 | Top-1: 0.0217 | Top-3: 0.0512 | Top-5: 0.0784 | F1: 0.0091
Val   | Loss: 6.1854 | Top-1: 0.0161 | Top-3: 0.0420 | Top-5: 0.0563 | F1: 0.0040
Learning rate: 0.00030000
Status: Saved new best model
Best Val F1 so far: 0.0040
Best Val Top-5 so far: 0.0563
Epochs without improvement: 0/15
\nEpoch 4/70
--------------------------------------------------------------------------------


Train | Loss: 5.7841 | Top-1: 0.0406 | Top-3: 0.0898 | Top-5: 0.1248 | F1: 0.0190
Val   | Loss: 5.9572 | Top-1: 0.0241 | Top-3: 0.0643 | Top-5: 0.0813 | F1: 0.0074
Learning rate: 0.00030000
Status: Saved new best model
Best Val F1 so far: 0.0074
Best Val Top-5 so far: 0.0813
Epochs without improvement: 0/15
\nEpoch 5/70
--------------------------------------------------------------------------------


...

Train | Loss: 0.8982 | Top-1: 0.9711 | Top-3: 0.9994 | Top-5: 1.0000 | F1: 0.9656
Val   | Loss: 4.5050 | Top-1: 0.2116 | Top-3: 0.3777 | Top-5: 0.4562 | F1: 0.1589
Learning rate: 0.00007500
Status: No improvement
Best Val F1 so far: 0.1706
Best Val Top-5 so far: 0.4634
Epochs without improvement: 11/15
\nEpoch 54/70
--------------------------------------------------------------------------------


Train | Loss: 0.8971 | Top-1: 0.9691 | Top-3: 0.9992 | Top-5: 0.9998 | F1: 0.9673
Val   | Loss: 4.4985 | Top-1: 0.2179 | Top-3: 0.3866 | Top-5: 0.4652 | F1: 0.1659
Learning rate: 0.00003750
Status: No improvement
Best Val F1 so far: 0.1706
Best Val Top-5 so far: 0.4634
Epochs without improvement: 12/15
\nEpoch 55/70
--------------------------------------------------------------------------------


Train | Loss: 0.8882 | Top-1: 0.9674 | Top-3: 0.9994 | Top-5: 1.0000 | F1: 0.9642
Val   | Loss: 4.4973 | Top-1: 0.2223 | Top-3: 0.3866 | Top-5: 0.4652 | F1: 0.1675
Learning rate: 0.00003750
Status: No improvement
Best Val F1 so far: 0.1706
Best Val Top-5 so far: 0.4634
Epochs without improvement: 13/15
\nEpoch 56/70
--------------------------------------------------------------------------------


Train | Loss: 0.8795 | Top-1: 0.9719 | Top-3: 0.9998 | Top-5: 1.0000 | F1: 0.9687
Val   | Loss: 4.4879 | Top-1: 0.2259 | Top-3: 0.3830 | Top-5: 0.4679 | F1: 0.1697
Learning rate: 0.00003750
Status: No improvement
Best Val F1 so far: 0.1706
Best Val Top-5 so far: 0.4634
Epochs without improvement: 14/15
\nEpoch 57/70
--------------------------------------------------------------------------------


Train | Loss: 0.8832 | Top-1: 0.9699 | Top-3: 0.9996 | Top-5: 1.0000 | F1: 0.9623
Val   | Loss: 4.5071 | Top-1: 0.2170 | Top-3: 0.3795 | Top-5: 0.4661 | F1: 0.1629
Learning rate: 0.00003750
Status: No improvement
Best Val F1 so far: 0.1706
Best Val Top-5 so far: 0.4634
Epochs without improvement: 15/15
\nEarly stopping triggered.
\nTraining time minutes: 15.75
Best model saved to: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_wlasl1000.pt


## 6. Save history and evaluate test set

In [10]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_top1, test_top3, test_top5, test_f1 = run_epoch(
    model, test_loader, optimizer=None, phase="Test", epoch=1, total_epochs=1
)

print("=" * 80)
print("WLASL1000 Test Evaluation")
print("=" * 80)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

RESULT_FILE = MODEL_DIR / f"bigru_attention_{PREFIX}_result_summary.csv"

result_df = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "model": "BiGRU + Temporal Attention",
    "clean_samples": len(df),
    "classes": NUM_CLASSES,
    "input_shape": f"(60, {INPUT_SIZE})",
    "velocity_features": USE_VELOCITY,
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "checkpoint_epoch": checkpoint["epoch"],
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_f1,
    "model_path": str(MODEL_PATH),
    "history_path": str(HISTORY_PATH)
}])

result_df.to_csv(RESULT_FILE, index=False)
print("Saved result summary:", RESULT_FILE)
result_df

WLASL1000 Test Evaluation
Test Loss: 4.5744
Test Top-1 Accuracy: 0.2080
Test Top-3 Accuracy: 0.3884
Test Top-5 Accuracy: 0.4634
Test Macro F1: 0.1674
Saved result summary: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_wlasl1000_result_summary.csv


,dataset,model,clean_samples,classes,input_shape,velocity_features,best_val_f1,best_val_top5,checkpoint_epoch,test_top1_accuracy,test_top3_accuracy,test_top5_accuracy,test_macro_f1,model_path,history_path
0,WLASL1000,BiGRU + Temporal Attention,7232,1000,"(60, 516)",True,0.170589,0.463393,42,0.208036,0.388393,0.463393,0.167371,E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attent...,E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attent...
